In [ ]:
# cell 0
# Environment setup & repo root resolution + load secrets (credentials.json)
# IMPORTANT: DB identity comes from secrets; Azure SQL publishing is optional

import json
import sys
from pathlib import Path

def _find_repo_root(start: Path) -> Path:
    """
    Find the repo root by walking upwards until we find expected repo markers.
    This makes notebook execution reliable even if cwd is not /notebooks.
    """
    start = start.resolve()
    candidates = [start] + list(start.parents)

    for p in candidates:
        if (p / "src").exists() and (p / "secrets").exists():
            return p

    raise FileNotFoundError(
        "Could not locate repo root. Expected to find both 'src/' and 'secrets/' "
        f"in one of the parent directories of: {start}"
    )

# Determine repo root (robust for VS Code, Jupyter, and different working directories)
NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = _find_repo_root(NOTEBOOK_CWD)

# Ensure repo modules are importable (src/ layout)
SRC_PATH = REPO_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

# Load secrets
SECRETS_PATH = REPO_ROOT / "secrets" / "credentials.json"
if not SECRETS_PATH.exists():
    raise FileNotFoundError(
        f"Secrets file not found at: {SECRETS_PATH}\n"
        "Create it locally (and ensure it is in .gitignore)."
    )

with open(SECRETS_PATH, "r", encoding="utf-8") as f:
    SECRETS = json.load(f)

# Validate required config blocks with clear messages
for key in ("betfair", "paths"):
    if key not in SECRETS:
        raise KeyError(
            f"Missing required '{key}' block in secrets file: {SECRETS_PATH}\n"
            "Expected at least: betfair, paths. Optional: user, azure_sql."
        )

BETFAIR = SECRETS["betfair"]
PATHS   = SECRETS["paths"]
USERCFG = SECRETS.get("user", {})
AZSQL   = SECRETS.get("azure_sql", None)

# Results directory (portable; configured in secrets)
RESULTS_CSV_DIR = Path(PATHS["results_csv_dir"])
RESULTS_CSV_DIR.mkdir(parents=True, exist_ok=True)

# DB write identity (comes from secrets; no hard-coded user identifiers in code)
DB_USER_ID = (USERCFG.get("db_user_id") or "").strip() or None

# Optional publishing toggle
ENABLE_AZURE_SQL = bool(USERCFG.get("enable_azure_sql", False))

# Safe-by-default: no writes unless explicitly flipped in secrets
DRY_RUN = bool(USERCFG.get("dry_run", True))

print("Secrets loaded OK")
print("Repo root:", REPO_ROOT)
print("CSV output dir:", RESULTS_CSV_DIR)
print("Azure SQL enabled:", ENABLE_AZURE_SQL)
print("DB_USER_ID set:", bool(DB_USER_ID))
print("DRY_RUN:", DRY_RUN)


In [ ]:
# cell 1
# Project imports (from src/ package) — required for repeatable execution

from betfair_results_downloader.csv_utils import (
    clean_and_remove_duplicates,
    update_csv_with_new_data,
)


In [ ]:
# cell 2
# Fetch Betfair cleared orders -> df_co (interactive login, no certs, paginated + retry/backoff)

import betfairlightweight
import pandas as pd
import datetime as dt
import json
import time
from zoneinfo import ZoneInfo
from betfairlightweight.exceptions import APIError

# --- Credentials (loaded in Cell 0) ---
my_username = BETFAIR.get("username", "")
my_password = BETFAIR.get("password", "")
my_app_key  = BETFAIR.get("app_key", "")

print("Betfair username loaded:", my_username if my_username else "(blank)")
print("Betfair app key loaded:", "YES" if my_app_key else "NO")

# Keep this reasonable for testing; you can increase later
LOOKBACK_DAYS = 30
print("LOOKBACK_DAYS:", LOOKBACK_DAYS)

utc_now = dt.datetime.now(dt.timezone.utc)
from_dt = (utc_now - dt.timedelta(days=LOOKBACK_DAYS)).strftime("%Y-%m-%dT%H:%M:%SZ")
to_dt   = utc_now.strftime("%Y-%m-%dT%H:%M:%SZ")
print("Settled date range UTC:", from_dt, "->", to_dt)

settled_range = betfairlightweight.filters.time_range(from_=from_dt, to=to_dt)

# --- Connect (interactive login: NO certs) ---
trading = betfairlightweight.APIClient(
    username=my_username,
    password=my_password,
    app_key=my_app_key
)
trading.login_interactive()

def call_list_cleared_orders(from_record: int, record_count: int, max_retries: int = 5):
    """
    Call listClearedOrders with retry/backoff for Betfair TIMEOUT_ERROR.
    """
    for attempt in range(1, max_retries + 1):
        try:
            return trading.betting.list_cleared_orders(
                bet_status="SETTLED",
                settled_date_range=settled_range,
                from_record=from_record,
                record_count=record_count
            )
        except APIError as e:
            msg = str(e)
            is_timeout = ("TIMEOUT_ERROR" in msg) or ("ANGX-0010" in msg)
            if not is_timeout or attempt == max_retries:
                raise
            sleep_s = min(2 ** attempt, 20)  # capped exponential backoff
            print(f"Timeout from Betfair (attempt {attempt}/{max_retries}). Retrying in {sleep_s}s...")
            time.sleep(sleep_s)

# --- Paginate in smaller chunks to avoid timeouts ---
PAGE_SIZE = 200  # smaller = more reliable on slow APING days
indexrecord = 0
all_rows = []

while True:
    cleared_orders = call_list_cleared_orders(indexrecord, PAGE_SIZE)

    # Parse via JSON to be compatible across betfairlightweight response modes
    data = json.loads(cleared_orders.json())
    batch = data.get("clearedOrders", [])
    if not batch:
        break

    all_rows.extend(batch)
    indexrecord += PAGE_SIZE

df_co = pd.DataFrame(all_rows)

# Ensure required columns exist for downstream cells
required_cols = [
    "eventTypeId", "eventId", "marketId", "selectionId", "handicap", "betId",
    "placedDate", "persistenceType", "orderType", "side", "betOutcome",
    "priceRequested", "settledDate", "lastMatchedDate", "betCount",
    "priceMatched", "priceReduced", "sizeSettled", "profit",
    "customerOrderRef", "customerStrategyRef"
]
for c in required_cols:
    if c not in df_co.columns:
        df_co[c] = pd.NA
df_co = df_co[required_cols]

# Win column
def determine_win(row):
    if (row["side"] == "BACK" and row["betOutcome"] == "LOST") or (row["side"] == "LAY" and row["betOutcome"] == "WON"):
        return 0
    return 1

if not df_co.empty:
    df_co["Win"] = df_co.apply(determine_win, axis=1)
else:
    df_co["Win"] = pd.Series(dtype="int")

# placedDate to Australia/Sydney
df_co["placedDate"] = pd.to_datetime(df_co["placedDate"], utc=True, errors="coerce")
aet_zone = ZoneInfo("Australia/Sydney")
df_co["placedDate"] = df_co["placedDate"].dt.tz_convert(aet_zone)
df_co["placedDateOnly"] = df_co["placedDate"].dt.date
df_co["placedTimeOnly"] = df_co["placedDate"].dt.time

print("Rows:", len(df_co))
print("Columns:", df_co.columns.tolist())


In [ ]:
# cell 3
# Enrich df_co with event/market metadata (list_market_catalogue)
# - portable outputs directory (repo-root relative)
# - deterministic column prefixes (no _x/_y collisions)
# - optional cache to speed repeat runs

import time
from pathlib import Path
import pandas as pd
from betfairlightweight import filters

assert "df_co" in globals(), "Expected df_co to exist. Run Cell 2 first."
assert "trading" in globals(), "Expected 'trading' betfair client to exist (created in Cell 2)."

# -----------------------------
# Settings
# -----------------------------
ENABLE_EVENT_META = True
EVENT_BATCH_SIZE = 50
SLEEP_SECONDS = 0.20

USE_CACHE = True
OUT_DIR = REPO_ROOT / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CACHE_PATH = OUT_DIR / "market_catalogue_event_cache.csv"
SNAPSHOT_PATH = OUT_DIR / "market_catalogue_event_latest.csv"

# -----------------------------
# Helpers
# -----------------------------
def chunked(seq, n):
    seq = list(seq)
    for i in range(0, len(seq), n):
        yield seq[i : i + n]

def _is_old_enrichment_col(c: str) -> bool:
    # Remove prior enrichment columns if they exist
    prefixes = (
        "mkt_", "evt_", "cmp_", "vn_", "crs_", "cat_",
    )
    if c.endswith(("_x", "_y")):
        return True
    return c.startswith(prefixes)

# -----------------------------
# Main
# -----------------------------
if not ENABLE_EVENT_META:
    print("[ENRICH] ENABLE_EVENT_META is False; skipping enrichment.")
else:
    # Drop any prior enrichment to avoid collisions
    cols_to_drop = [c for c in df_co.columns if _is_old_enrichment_col(c)]
    if cols_to_drop:
        print(f"[ENRICH] Dropping {len(cols_to_drop)} prior enrichment columns to avoid collisions.")
        df_co = df_co.drop(columns=cols_to_drop)

    # Ensure ids are str for joins
    df_co["marketId"] = df_co["marketId"].astype(str)

    # If cache exists, load it
    if USE_CACHE and CACHE_PATH.exists():
        df_cache = pd.read_csv(CACHE_PATH, dtype=str)
        print(f"[ENRICH] Loaded cache: {CACHE_PATH} | rows={len(df_cache):,}")
    else:
        df_cache = pd.DataFrame()

    # MarketIds to fetch
    unique_market_ids = sorted(df_co["marketId"].dropna().unique().tolist())
    cached_market_ids = set(df_cache["marketId"].unique()) if (not df_cache.empty and "marketId" in df_cache.columns) else set()
    missing_market_ids = [m for m in unique_market_ids if m not in cached_market_ids]

    print(f"[ENRICH] Unique marketIds in df_co: {len(unique_market_ids):,}")
    print(f"[ENRICH] Cache rows: {len(df_cache):,}")
    print(f"[ENRICH] Missing marketIds to fetch: {len(missing_market_ids):,}")

    fetched_rows = []

    for batch in chunked(missing_market_ids, EVENT_BATCH_SIZE):
        time.sleep(SLEEP_SECONDS)

        cats = trading.betting.list_market_catalogue(
            filter=filters.market_filter(market_ids=batch),
            max_results=1000,
            market_projection=["MARKET_START_TIME", "EVENT"],
        )

        for cat in cats:
            fetched_rows.append({
                "marketId": str(cat.market_id),
                "mkt_marketName": getattr(cat, "market_name", None),
                "mkt_marketStartTime": getattr(cat, "market_start_time", None),
                "evt_eventId": str(cat.event.id) if getattr(cat, "event", None) else None,
                "evt_eventName": cat.event.name if getattr(cat, "event", None) else None,
                "evt_countryCode": cat.event.country_code if getattr(cat, "event", None) else None,
            })

    df_fetched = pd.DataFrame(fetched_rows)

    # Combine cache + fetched
    if not df_fetched.empty:
        df_new_cache = pd.concat([df_cache, df_fetched], ignore_index=True) if not df_cache.empty else df_fetched
        df_new_cache = df_new_cache.drop_duplicates(subset=["marketId"], keep="last")
        df_new_cache.to_csv(CACHE_PATH, index=False)
        df_new_cache.to_csv(SNAPSHOT_PATH, index=False)
        print(f"[ENRICH] Cache written: {CACHE_PATH}")
        print(f"[ENRICH] Snapshot written: {SNAPSHOT_PATH}")
    else:
        df_new_cache = df_cache
        if not df_cache.empty:
            df_cache.to_csv(SNAPSHOT_PATH, index=False)
            print(f"[ENRICH] No new rows fetched; snapshot written: {SNAPSHOT_PATH}")

    # Join onto df_co
    if not df_new_cache.empty:
        df_co = df_co.merge(df_new_cache, on="marketId", how="left")
        print(f"[ENRICH] Joined metadata. df_co cols={len(df_co.columns)} rows={len(df_co):,}")
    else:
        print("[ENRICH] No cache/fetch data available; skipping join.")


In [ ]:
# cell 4
# Data preview & validation (post-fetch, post-enrichment)

import pandas as pd

assert "df_co" in globals(), "df_co not found. Run Cell 2 (and Cell 3 if enrichment enabled)."

print("df_co rows:", len(df_co))
print("df_co cols:", len(df_co.columns))

# Core columns sanity check
core_cols = ["eventTypeId", "marketId", "betId", "profit", "placedDate"]
missing = [c for c in core_cols if c not in df_co.columns]
if missing:
    raise KeyError(f"Missing required columns in df_co: {missing}")

print("eventTypeId counts:")
print(df_co["eventTypeId"].value_counts(dropna=False).head(20))

display(df_co.head(10))


In [ ]:
# cell 5
# Write snapshot CSV

from datetime import datetime

csv_out = RESULTS_CSV_DIR / f"cleared_orders_cleaned_{datetime.now():%Y-%m-%d}.csv"
df_co.to_csv(csv_out, index=False)
print(f"Wrote: {csv_out}")


In [ ]:
# cell 6
# Checkpoint: confirm project utilities are available (imported in Cell 1)

assert "clean_and_remove_duplicates" in globals(), "Expected clean_and_remove_duplicates from Cell 1 import."
assert "update_csv_with_new_data" in globals(), "Expected update_csv_with_new_data from Cell 1 import."

print("✅ Project utilities available (csv_utils).")


In [ ]:
# cell 7
# Define canonical CSV output paths (portable; uses secrets-defined directory)

from pathlib import Path
import datetime

# RESULTS_CSV_DIR is defined in Cell 0 from secrets/credentials.json
# e.g. PATHS["results_csv_dir"]
csv_output_dir = Path(RESULTS_CSV_DIR)
csv_output_dir.mkdir(parents=True, exist_ok=True)

# Canonical "always the same" file used for idempotent updates
csv_path = csv_output_dir / "cleared_orders_cleaned.csv"

# Optional dated snapshot for audit / rollback
today_str = datetime.date.today().isoformat()  # YYYY-MM-DD
csv_snapshot_path = csv_output_dir / f"cleared_orders_cleaned_{today_str}.csv"

print("CSV output dir:", str(csv_output_dir))
print("Canonical CSV path:", str(csv_path))
print("Snapshot CSV path:", str(csv_snapshot_path))
print("Canonical exists:", csv_path.exists())


In [ ]:
# cell 8
# Update / append closed orders CSV (idempotent), then write a dated snapshot

import pandas as pd

# Update canonical file idempotently
update_csv_with_new_data(str(csv_path), df_co)

# Snapshot the canonical file (so snapshot always reflects final canonical content)
df_canonical = pd.read_csv(csv_path)
df_canonical.to_csv(csv_snapshot_path, index=False)

print(f"Wrote canonical: {csv_path}")
print(f"Wrote snapshot : {csv_snapshot_path}")
print("Rows in canonical:", len(df_canonical))


In [ ]:
# cell 9
# Azure upload dataset (Horse + Greyhound only) + market-level aggregation
# NOTE: We explicitly coerce eventTypeId to numeric to avoid dtype-mismatch (e.g., "7" vs 7).

import pandas as pd
from datetime import datetime

assert "df_co" in globals(), "df_co not found. Run Cell 2 first."

def log(msg: str):
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}")

ALLOWED_EVENT_TYPE_IDS = {7, 4339}  # 7 = Horse Racing, 4339 = Greyhound Racing

# --- sanity checks (up front, clear errors) ---
required_cols = {"eventTypeId", "marketId", "profit", "betId", "placedDate"}
missing = required_cols - set(df_co.columns)
if missing:
    raise KeyError(f"Missing required columns in df_co: {sorted(missing)}")

# --- normalize eventTypeId dtype (robust across CSV reads/merges) ---
df_stage = df_co.copy()
df_stage["eventTypeId"] = pd.to_numeric(df_stage["eventTypeId"], errors="coerce").astype("Int64")

# --- filter for Azure ---
rows_before = len(df_stage)
counts_before = df_stage["eventTypeId"].value_counts(dropna=False)

df_azure_upload = df_stage[df_stage["eventTypeId"].isin(ALLOWED_EVENT_TYPE_IDS)].copy()

rows_after = len(df_azure_upload)
log(f"Rows before Azure filter: {rows_before:,}")
log(f"Rows after  Azure filter: {rows_after:,}")
log(f"Allowed eventTypeIds: {sorted(ALLOWED_EVENT_TYPE_IDS)}")
log(f"eventTypeId dtype after coercion: {df_stage['eventTypeId'].dtype}")

# Report excluded ids (useful audit trail)
excluded_counts = counts_before[~counts_before.index.isin(list(ALLOWED_EVENT_TYPE_IDS))]
if not excluded_counts.empty:
    print("[AZURE FILTER] Excluded eventTypeIds (rows dropped):")
    print(excluded_counts.to_string())

# Hard stop if empty (prevents accidental empty uploads)
if rows_after == 0:
    raise ValueError(
        "[AZURE FILTER] No rows remain after filtering to allowed eventTypeIds. "
        "This usually indicates a dtype mismatch or unexpected eventTypeIds."
    )

print("[AZURE FILTER] Included eventTypeIds (rows kept):")
print(df_azure_upload["eventTypeId"].value_counts(dropna=False).to_string())

# --- aggregate ONLY from filtered df ---
df_market_results = (
    df_azure_upload
    .groupby("marketId", as_index=False)
    .agg(
        Profit=("profit", "sum"),
        Bets=("betId", "count"),
        FirstPlaced=("placedDate", "min"),
        LastPlaced=("placedDate", "max"),
    )
)

log(f"df_market_results shape: {df_market_results.shape[0]:,} rows x {df_market_results.shape[1]} cols")
display(df_market_results.head(10))


In [ ]:
# cell 10
# Build rows_to_write for Azure SQL (NO user id embedded) — round Profit to 2dp

from decimal import Decimal, ROUND_HALF_UP

assert "df_market_results" in globals(), "df_market_results not found. Run Cell 9 first."

def money2(x) -> Decimal:
    return Decimal(str(x)).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)

rows_to_write = []
for _, r in df_market_results.iterrows():
    market_id = Decimal(str(r["marketId"]))
    profit    = money2(r["Profit"])
    rows_to_write.append((market_id, profit, ""))

print("Prepared rows_to_write (market-level, no user id)")
print("Rows to write:", len(rows_to_write))
print("Sample (first 10):")
for row in rows_to_write[:10]:
    print(row)


In [ ]:
# cell 11
# Azure SQL pre-check (READ ONLY): row count + duplicates + profit variance

import pyodbc

assert ENABLE_AZURE_SQL is True, "Azure SQL is disabled. Set user.enable_azure_sql=true in secrets."
assert DB_USER_ID is not None, "DB user id missing. Set user.db_user_id in secrets."
assert AZSQL is not None, "azure_sql block missing in secrets."

az_conn_str = (
    f"DRIVER={{{AZSQL['driver']}}};"
    f"SERVER={AZSQL['server']},{AZSQL['port']};"
    f"DATABASE={AZSQL['database']};"
    f"UID={AZSQL['username']};"
    f"PWD={AZSQL['password']};"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
    "Connection Timeout=30;"
)

conn = None
cur = None

try:
    print("Testing Azure SQL connection (read-only)...")
    conn = pyodbc.connect(az_conn_str)
    cur = conn.cursor()
    cur.execute("SELECT 1;")
    print("Azure SQL connection OK.")

    cur.execute("SELECT COUNT(*) FROM dbo.MarketResults WHERE RTRIM(UserID) = ?;", (DB_USER_ID,))
    n = cur.fetchone()[0]
    print(f"Existing MarketResults rows for {DB_USER_ID}: {n}")

    sql = """
    SELECT
      RTRIM(UserID) AS UserID,
      MarketID,
      COUNT(*) AS cnt,
      MIN(Profit) AS min_profit,
      MAX(Profit) AS max_profit
    FROM dbo.MarketResults
    WHERE RTRIM(UserID) = ?
    GROUP BY RTRIM(UserID), MarketID
    HAVING COUNT(*) > 1
    ORDER BY cnt DESC;
    """
    cur.execute(sql, (DB_USER_ID,))
    rows = cur.fetchall()

    if not rows:
        print("No duplicates found for (UserID, MarketID).")
    else:
        print(f"WARNING: duplicates found for (UserID, MarketID): {len(rows)} groups. Showing up to 20:")
        for r in rows[:20]:
            print(r)

        conflicting = sum(1 for r in rows if r[3] != r[4])
        print(f"Conflicting duplicate groups (min_profit != max_profit): {conflicting} of {len(rows)}")

finally:
    if cur is not None:
        cur.close()
    if conn is not None:
        conn.close()


In [ ]:
# cell 12
# Azure SQL: post-write row count check (read-only, pyodbc)

import pyodbc

assert ENABLE_AZURE_SQL is True, "Azure SQL is disabled. Set user.enable_azure_sql=true in secrets."
assert DB_USER_ID is not None, "DB user id missing. Set user.db_user_id in secrets."
assert AZSQL is not None, "azure_sql block missing in secrets."

az_conn_str = (
    f"DRIVER={{{AZSQL['driver']}}};"
    f"SERVER={AZSQL['server']},{AZSQL['port']};"
    f"DATABASE={AZSQL['database']};"
    f"UID={AZSQL['username']};"
    f"PWD={AZSQL['password']};"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
    "Connection Timeout=30;"
)

conn = None
cur = None

try:
    print("Testing Azure SQL connection (read-only)...")
    conn = pyodbc.connect(az_conn_str)
    cur = conn.cursor()

    cur.execute("SELECT COUNT(*) FROM dbo.MarketResults WHERE RTRIM(UserID) = ?;", (DB_USER_ID,))
    n = cur.fetchone()[0]
    print(f"Azure: rows for DB_USER_ID={DB_USER_ID!r}: {n}")

finally:
    if cur is not None:
        cur.close()
    if conn is not None:
        conn.close()


In [ ]:
# cell 13
# Azure SQL: rebuild MarketResults for DB_USER_ID (DELETE then INSERT)

import pyodbc

assert ENABLE_AZURE_SQL is True, "Azure SQL is disabled. Set user.enable_azure_sql=true in secrets."
assert DB_USER_ID is not None, "DB user id missing. Set user.db_user_id in secrets."
assert AZSQL is not None, "azure_sql block missing in secrets."
assert DRY_RUN is False, "DRY_RUN must be False to write to Azure SQL."
assert "rows_to_write" in globals() and len(rows_to_write) > 0, "rows_to_write not prepared. Run Cell 10."

az_conn_str = (
    f"DRIVER={{{AZSQL['driver']}}};"
    f"SERVER={AZSQL['server']},{AZSQL['port']};"
    f"DATABASE={AZSQL['database']};"
    f"UID={AZSQL['username']};"
    f"PWD={AZSQL['password']};"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
    "Connection Timeout=30;"
)

conn = None
cur = None

try:
    print(f"Rebuilding dbo.MarketResults for DB_USER_ID={DB_USER_ID!r} ...")
    conn = pyodbc.connect(az_conn_str)
    conn.autocommit = False
    cur = conn.cursor()

    cur.execute("DELETE FROM dbo.MarketResults WHERE RTRIM(UserID) = ?;", (DB_USER_ID,))
    print("Deleted rows:", cur.rowcount)

    # Add DB_USER_ID at write time (not stored in rows_to_write)
    rows_for_db = [(DB_USER_ID, *row) for row in rows_to_write]

    cur.fast_executemany = True
    cur.executemany(
        "INSERT INTO dbo.MarketResults (UserID, MarketID, Profit, Notes) VALUES (?, ?, ?, ?);",
        rows_for_db
    )

    conn.commit()
    print("✅ Commit complete.")

finally:
    if cur is not None:
        cur.close()
    if conn is not None:
        conn.close()
